# H2O-style GSM8K Accuracy vs KV Cache Budget

This Colab notebook evaluates three policies on GSM8K:

1. **Local** — same definition as the H2O paper Fig. 4 **“Local”** curve: the cache retains **only the most recent KV embeddings** (a trailing window; no heavy-hitters). Here that is **`H2OQwen3_5ForCausalLM`** in **absolute** mode with **`hh_size=0`** and **`recent_size`** = the budget fraction of the full-KV reference `MAX_CONTEXT_LENGTH` (= max prompt tokens on the **eval split** + `MAX_NEW_TOKENS`). **Budget %** sweeps the size of that recent window.
2. **Full** — **`H2OQwen3_5ForCausalLM`** in **ratio mode** at **100%** of that same `MAX_CONTEXT_LENGTH` reference (H2O at max budget; comparable baseline when contrasted with a full-cache run if you add one).
3. **H2O** — same H2O wrapper with **swept** `hh_ratio` / `h2o_full_cache_size` for each budget on the x-axis.

**Budget %:** each point is a fraction of **`max_prompt_tokens + MAX_NEW_TOKENS`** over the loaded `test_data` (respects `LIMIT`), not a fixed 2048 and not prompt-only.

Default checkpoint: **`Qwen/Qwen3.5-0.8B`** (dense Qwen3.5; H2O path matches `modify_qwen.py`).

It saves `results.csv`, `predictions.csv`, `results.json`, and `accuracy_vs_budget.png`.

Upload **`modify_qwen.py`** to `/content/modify_qwen.py` before running **Local**, **Full**, or **H2O** cells.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
!pip -q install -U "transformers>=4.53.0" datasets accelerate matplotlib pandas tqdm

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 377, in run
    requirement_set = resolver.resolve(
                      ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/resolution/resolvelib/resolver.py", line 95, in resolve
    result = self._result = resolver.resolve(
                            ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_vendor/resolvelib/resolvers.py", line 546, in resolve
    state = resolution.resolve(requirements, max_rounds=max_rounds)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [3]:
import os, re, gc, json, random, importlib.util
from pathlib import Path

import torch
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig

SEED = 3407
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

torch: 2.10.0+cu128
cuda available: True
gpu: Tesla T4


In [4]:
# =========================
# Experiment configuration
# =========================

MODEL_NAME_OR_PATH = "Qwen/Qwen3.5-0.8B"  # Dense Qwen3.5 (matches modify_qwen H2O wrapper)
H2O_MODULE_PATH = "/content/modify_qwen.py"         # Upload your fixed file here

OUTPUT_DIR = Path("/content/h2o_gsm8k_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BUDGETS = [4, 8, 12, 16, 20, 30, 40, 60, 80, 100]

# Start small for debugging. Set LIMIT=None for full GSM8K test split.
LIMIT = 50

BATCH_SIZE = 1
MAX_NEW_TOKENS = 256
# MAX_CONTEXT_LENGTH is set in the tokenizer/model cell after the tokenizer loads:
#   max_prompt_tokens = max over test_data["question"] (with build_prompt + chat template),
#   MAX_CONTEXT_LENGTH = max_prompt_tokens + MAX_NEW_TOKENS  (paper-like full KV length).
# Run order: gsm8k/test_data -> build_prompt -> tokenizer cell -> budget_to_kv_settings -> sweep.

DTYPE = "bf16"  # "bf16", "fp16", or "fp32"
USE_CHAT_TEMPLATE = True
DISABLE_THINKING = True

POLICIES = ["local", "full", "h2o"]


In [6]:
def get_torch_dtype(dtype_name: str):
    dtype_name = dtype_name.lower()
    if dtype_name == "bf16":
        return torch.bfloat16
    if dtype_name == "fp16":
        return torch.float16
    if dtype_name == "fp32":
        return torch.float32
    raise ValueError(f"Unknown dtype: {dtype_name}")

TORCH_DTYPE = get_torch_dtype(DTYPE)

def cleanup_model(model=None):
    if model is not None:
        del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def load_h2o_class(module_path: str):
    module_path = Path(module_path)
    if not module_path.exists():
        raise FileNotFoundError(
            f"Could not find {module_path}. Upload your fixed modify_qwen.py to this path."
        )
    spec = importlib.util.spec_from_file_location("modify_qwen", str(module_path))
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module.H2OQwen3_5ForCausalLM

H2OQwen3_5ForCausalLM = None
if any(p in POLICIES for p in ("local", "full", "h2o")):
    H2OQwen3_5ForCausalLM = load_h2o_class(H2O_MODULE_PATH)
    print("Loaded:", H2OQwen3_5ForCausalLM)
else:
    print("Skipping H2O class load (POLICIES has neither 'local', 'full', nor 'h2o').")


Loaded: <class 'modify_qwen.H2OQwen3_5ForCausalLM'>


In [7]:
gsm8k = load_dataset("openai/gsm8k", "main")
test_data = gsm8k["test"]

if LIMIT is not None:
    test_data = test_data.select(range(min(LIMIT, len(test_data))))

print(test_data)
print(test_data[0])

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Dataset({
    features: ['question', 'answer'],
    num_rows: 50
})
{'question': "Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?", 'answer': 'Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.\n#### 18'}


In [8]:
_NUMBER_RE = re.compile(r"[-+]?\d[\d,]*(?:\.\d+)?")

def normalize_number(s):
    if s is None:
        return None
    s = str(s).strip().replace(",", "")
    s = re.sub(r"[\.$%\s]+$", "", s)
    try:
        x = float(s)
        if abs(x - int(x)) < 1e-9:
            return str(int(x))
        return str(x)
    except Exception:
        return s

def extract_gold_answer(answer_text):
    if "####" in answer_text:
        return normalize_number(answer_text.split("####")[-1].strip())
    nums = _NUMBER_RE.findall(answer_text)
    return normalize_number(nums[-1]) if nums else None

def extract_gsm8k_answer(text):
    if text is None:
        return None

    patterns = [
        r"####\s*([-+]?\d[\d,]*(?:\.\d+)?)",
        r"(?:final answer|answer is|answer:)\s*([-+]?\d[\d,]*(?:\.\d+)?)",
    ]
    for pat in patterns:
        matches = re.findall(pat, text, flags=re.IGNORECASE)
        if matches:
            return normalize_number(matches[-1])

    nums = _NUMBER_RE.findall(text)
    return normalize_number(nums[-1]) if nums else None

for s in ["#### 72", "The answer is 1,234.", "Final answer: 72.0", "x 3 then 5"]:
    print(s, "->", extract_gsm8k_answer(s))

#### 72 -> 72
The answer is 1,234. -> 1234
Final answer: 72.0 -> 72
x 3 then 5 -> 5


In [10]:
def build_prompt(question: str) -> str:
    if DISABLE_THINKING:
        return (
            f"Question: {question}\n\n"
            "You are solving a grade-school math word problem.\n"
            "- You may use a tiny amount of scratch work (at most a few short lines).\n"
            "- Do NOT write a long chain-of-thought or step-by-step essay.\n"
            "- Your last line MUST be exactly: #### <integer>\n"
            "  where <integer> is the final numeric answer only (no units, no punctuation after the number).\n"
        )
    return (
        f"Question: {question}\n\n"
        "You are solving a grade-school math word problem.\n"
        "- You may show brief reasoning or steps if helpful.\n"
        "- End your response with a line exactly: #### <integer>\n"
        "  where <integer> is the final numeric answer only.\n"
    )

def tokenize_prompts(tokenizer, prompts, device):
    if USE_CHAT_TEMPLATE:
        encoded = []
        for prompt in prompts:
            messages = [{"role": "user", "content": prompt}]
            try:
                enc = tokenizer.apply_chat_template(
                    messages,
                    tokenize=True,
                    add_generation_prompt=True,
                    enable_thinking=not DISABLE_THINKING,
                    return_dict=True,
                    return_tensors="pt",
                )
            except TypeError:
                enc = tokenizer.apply_chat_template(
                    messages,
                    tokenize=True,
                    add_generation_prompt=True,
                    return_dict=True,
                    return_tensors="pt",
                )
            encoded.append(enc)

        input_ids_list = [e["input_ids"][0] for e in encoded]
        max_len = max(x.shape[0] for x in input_ids_list)
        pad_id = tokenizer.pad_token_id

        padded, attn = [], []
        for ids in input_ids_list:
            pad_len = max_len - ids.shape[0]
            if tokenizer.padding_side == "left":
                padded_ids = torch.cat([torch.full((pad_len,), pad_id, dtype=ids.dtype), ids])
                mask = torch.cat([torch.zeros(pad_len, dtype=torch.long), torch.ones(ids.shape[0], dtype=torch.long)])
            else:
                padded_ids = torch.cat([ids, torch.full((pad_len,), pad_id, dtype=ids.dtype)])
                mask = torch.cat([torch.ones(ids.shape[0], dtype=torch.long), torch.zeros(pad_len, dtype=torch.long)])
            padded.append(padded_ids)
            attn.append(mask)

        return {
            "input_ids": torch.stack(padded, dim=0).to(device),
            "attention_mask": torch.stack(attn, dim=0).to(device),
        }

    return tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_CONTEXT_LENGTH,
    ).to(device)


In [11]:
def load_tokenizer(model_name_or_path):
    tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, trust_remote_code=True)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    return tokenizer


def force_qwen_eager_attention(config):
    if hasattr(config, "_flash_attn_2_enabled"):
        config._flash_attn_2_enabled = False
    if hasattr(config, "attn_implementation"):
        config.attn_implementation = "eager"
    return config


def load_full_model(model_name_or_path):
    config = AutoConfig.from_pretrained(model_name_or_path, trust_remote_code=True)
    force_qwen_eager_attention(config)
    if hasattr(config, "text_config") and config.text_config is not None:
        force_qwen_eager_attention(config.text_config)
    try:
        model = AutoModelForCausalLM.from_pretrained(
            model_name_or_path,
            config=config,
            torch_dtype=TORCH_DTYPE,
            device_map="auto",
            trust_remote_code=True,
            attn_implementation="eager",
        )
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(
            model_name_or_path,
            config=config,
            torch_dtype=TORCH_DTYPE,
            device_map="auto",
            trust_remote_code=True,
        )
    model.eval()
    return model


def load_h2o_model_ratio(model_name_or_path, hh_ratio: float, full_cache_size: int):
    config = AutoConfig.from_pretrained(model_name_or_path, trust_remote_code=True)
    force_qwen_eager_attention(config)
    if hasattr(config, "text_config") and config.text_config is not None:
        force_qwen_eager_attention(config.text_config)
    config.hh_ratio = float(hh_ratio)
    config.h2o_full_cache_size = int(full_cache_size)
    tc = getattr(config, "text_config", None)
    if tc is not None:
        tc.hh_ratio = float(hh_ratio)
        tc.h2o_full_cache_size = int(full_cache_size)

    try:
        model = H2OQwen3_5ForCausalLM.from_pretrained(
            model_name_or_path,
            config=config,
            torch_dtype=TORCH_DTYPE,
            device_map="auto",
            trust_remote_code=True,
            attn_implementation="eager",
        )
    except TypeError:
        model = H2OQwen3_5ForCausalLM.from_pretrained(
            model_name_or_path,
            config=config,
            torch_dtype=TORCH_DTYPE,
            device_map="auto",
            trust_remote_code=True,
        )
    model.eval()
    return model


def load_h2o_model_absolute(model_name_or_path, hh_size: int, recent_size: int):
    config = AutoConfig.from_pretrained(model_name_or_path, trust_remote_code=True)
    force_qwen_eager_attention(config)
    if hasattr(config, "text_config") and config.text_config is not None:
        force_qwen_eager_attention(config.text_config)

    # Absolute recent-window mode: unset ratio-mode fields so H2OKVCache uses hh_size / recent_size.
    config.hh_ratio = None
    config.h2o_full_cache_size = None
    config.hh_size = int(hh_size)
    config.recent_size = int(recent_size)
    tc = getattr(config, "text_config", None)
    if tc is not None:
        tc.hh_ratio = None
        tc.h2o_full_cache_size = None
        tc.hh_size = int(hh_size)
        tc.recent_size = int(recent_size)

    try:
        model = H2OQwen3_5ForCausalLM.from_pretrained(
            model_name_or_path,
            config=config,
            torch_dtype=TORCH_DTYPE,
            device_map="auto",
            trust_remote_code=True,
            attn_implementation="eager",
        )
    except TypeError:
        model = H2OQwen3_5ForCausalLM.from_pretrained(
            model_name_or_path,
            config=config,
            torch_dtype=TORCH_DTYPE,
            device_map="auto",
            trust_remote_code=True,
        )
    model.eval()
    return model


tokenizer = load_tokenizer(MODEL_NAME_OR_PATH)
print("pad token:", tokenizer.pad_token, tokenizer.pad_token_id)
print("eos token:", tokenizer.eos_token, tokenizer.eos_token_id)


def _flat_token_id_len(ids) -> int:
    """Single-sequence token count; supports tensors, BatchEncoding, nested list chunks."""
    # apply_chat_template(..., return_dict=True) may nest input_ids as BatchEncoding in some versions.
    while (
        ids is not None
        and not isinstance(ids, torch.Tensor)
        and not isinstance(ids, (list, tuple))
        and hasattr(ids, "__contains__")
        and "input_ids" in ids
    ):
        ids = ids["input_ids"]

    if ids is None:
        return 0
    if isinstance(ids, torch.Tensor):
        return int(ids.numel())
    if isinstance(ids, (list, tuple)):
        if len(ids) == 0:
            return 0
        if isinstance(ids[0], (list, tuple)):
            return sum(len(part) for part in ids)
        return len(ids)
    raise TypeError(f"Unexpected token ids type: {type(ids)}")


def get_prompt_token_length(tokenizer, question: str) -> int:
    prompt = build_prompt(question)

    if USE_CHAT_TEMPLATE:
        messages = [{"role": "user", "content": prompt}]
        try:
            enc = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                enable_thinking=not DISABLE_THINKING,
                return_dict=True,
                return_tensors=None,
            )
        except TypeError:
            enc = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_dict=True,
                return_tensors=None,
            )

        return _flat_token_id_len(enc)

    enc = tokenizer(
        prompt,
        truncation=False,
        add_special_tokens=True,
    )
    return _flat_token_id_len(enc)


if len(test_data["question"]) == 0:
    raise ValueError("test_data has no rows; cannot compute MAX_CONTEXT_LENGTH")

max_prompt_tokens = max(
    get_prompt_token_length(tokenizer, q) for q in test_data["question"]
)
MAX_CONTEXT_LENGTH = int(max_prompt_tokens) + int(MAX_NEW_TOKENS)
print("max_prompt_tokens (eval set):", max_prompt_tokens)
print("MAX_CONTEXT_LENGTH (= max_prompt + MAX_NEW_TOKENS):", MAX_CONTEXT_LENGTH)


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

pad token: <|endoftext|> 248044
eos token: <|im_end|> 248046


In [ ]:
def budget_to_kv_settings(budget_percent: int, max_context_length: int):
    # max_context_length = paper full-KV reference for this eval: max_prompt_tokens(eval) + MAX_NEW_TOKENS.
    # budget_percent of that reference -> total_budget token slots; split evenly into HH + recent via ratio mode.
    total_budget = max(1, int((budget_percent / 100.0) * max_context_length))
    hh_ratio = (total_budget / 2.0) / float(max_context_length)
    hh_ratio = min(float(hh_ratio), 0.5)
    slot = max(1, int(max_context_length * hh_ratio))
    approx_effective_kv = 2 * slot
    return {
        "hh_ratio": hh_ratio,
        "full_cache_size": max_context_length,
        "total_budget": total_budget,
        "approx_effective_kv": approx_effective_kv,
        "approx_slot": slot,
    }


for b in [20, 100]:
    print("budget", b, "->", budget_to_kv_settings(b, MAX_CONTEXT_LENGTH))

def budget_to_local_recent(budget_percent: int, full_kv_ref: int):
    total_budget = max(1, int((budget_percent / 100.0) * full_kv_ref))
    return {"hh_size": 0, "recent_size": total_budget, "total_budget": total_budget}


for b in [20, 100]:
    print("local recent-only", b, "% ->", budget_to_local_recent(b, MAX_CONTEXT_LENGTH))


In [13]:
@torch.no_grad()
def evaluate_model(
    model,
    tokenizer,
    data,
    policy: str,
    budget_percent,
    hh_size=None,
    recent_size=None,
    hh_ratio=None,
    h2o_full_cache_size=None,
    approx_effective_kv=None,
):
    rows = []
    correct = 0
    total = 0

    device = model.device if hasattr(model, "device") else next(model.parameters()).device

    for start in tqdm(range(0, len(data), BATCH_SIZE), desc=f"{policy} budget={budget_percent}"):
        batch_items = data[start : start + BATCH_SIZE]
        questions = batch_items["question"]
        gold_texts = batch_items["answer"]
        prompts = [build_prompt(q) for q in questions]
        gold_answers = [extract_gold_answer(a) for a in gold_texts]

        inputs = tokenize_prompts(tokenizer, prompts, device)

        if hasattr(model, "reset_h2o_state"):
            model.reset_h2o_state()

        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

        for i, full_text in enumerate(decoded):
            pred = extract_gsm8k_answer(full_text)
            gold = gold_answers[i]
            is_correct = (pred is not None and gold is not None and pred == gold)
            correct += int(is_correct)
            total += 1

            rows.append({
                "policy": policy,
                "budget_percent": budget_percent,
                "hh_ratio": hh_ratio,
                "h2o_full_cache_size": h2o_full_cache_size,
                "approx_effective_kv": approx_effective_kv,
                "hh_size": hh_size,
                "recent_size": recent_size,
                "question": questions[i],
                "gold": gold,
                "pred": pred,
                "correct": bool(is_correct),
                "output": full_text,
            })

            if total <= 3:
                print("\n--- example ---")
                print("Q:", questions[i][:300])
                print("gold:", gold, "pred:", pred, "correct:", is_correct)
                print("output tail:", full_text[-500:])

    acc = correct / max(total, 1)
    summary = {
        "policy": policy,
        "budget_percent": budget_percent,
        "hh_ratio": hh_ratio,
        "h2o_full_cache_size": h2o_full_cache_size,
        "approx_effective_kv": approx_effective_kv,
        "hh_size": hh_size,
        "recent_size": recent_size,
        "accuracy": acc,
        "num_correct": correct,
        "num_total": total,
    }
    return summary, rows


In [ ]:
all_summaries = []
all_prediction_rows = []


def save_results():
    pd.DataFrame(all_summaries).to_csv(OUTPUT_DIR / "results.csv", index=False)
    pd.DataFrame(all_prediction_rows).to_csv(OUTPUT_DIR / "predictions.csv", index=False)
    with open(OUTPUT_DIR / "results.json", "w") as f:
        json.dump(all_summaries, f, indent=2)


def fan_out_predictions(rows, budget_values):
    # One eval run, many x-axis budget points: duplicate rows with matching budget_percent.
    out = []
    for b in budget_values:
        for r in rows:
            rr = dict(r)
            rr["budget_percent"] = b
            out.append(rr)
    return out


if "local" in POLICIES:
    for budget in BUDGETS:
        kv_loc = budget_to_local_recent(budget, MAX_CONTEXT_LENGTH)
        cleanup_model()
        print(
            f"\nLoading Local (paper Fig. 4 style — most recent KV only): budget={budget}% "
            f"hh_size={kv_loc['hh_size']} recent_size={kv_loc['recent_size']} "
            f"(total_budget_tokens={kv_loc['total_budget']})"
        )
        local_model = load_h2o_model_absolute(
            MODEL_NAME_OR_PATH, kv_loc["hh_size"], kv_loc["recent_size"]
        )
        local_summary, local_rows = evaluate_model(
            local_model,
            tokenizer,
            test_data,
            policy="local",
            budget_percent=budget,
            hh_size=0,
            recent_size=kv_loc["recent_size"],
            hh_ratio=None,
            h2o_full_cache_size=None,
            approx_effective_kv=kv_loc["total_budget"],
        )
        local_summary["total_budget_tokens"] = kv_loc["total_budget"]
        print("Local summary:", local_summary)
        all_summaries.append(local_summary)
        all_prediction_rows.extend(local_rows)
        save_results()
        cleanup_model(local_model)

if "full" in POLICIES:
    kv_full = budget_to_kv_settings(100, MAX_CONTEXT_LENGTH)
    cleanup_model()
    print(
        f"\nLoading Full (H2O @ 100% budget): hh_ratio={kv_full['hh_ratio']:.6g} "
        f"h2o_full_cache_size={kv_full['full_cache_size']} (~{kv_full['approx_effective_kv']} KV slots)"
    )
    full_model = load_h2o_model_ratio(MODEL_NAME_OR_PATH, kv_full["hh_ratio"], kv_full["full_cache_size"])
    full_summary, full_rows = evaluate_model(
        full_model,
        tokenizer,
        test_data,
        policy="full",
        budget_percent=100,
        hh_size=None,
        recent_size=None,
        hh_ratio=kv_full["hh_ratio"],
        h2o_full_cache_size=kv_full["full_cache_size"],
        approx_effective_kv=kv_full["approx_effective_kv"],
    )
    full_summary["total_budget_tokens"] = kv_full["total_budget"]
    print("Full summary:", full_summary)
    for b in BUDGETS:
        row = dict(full_summary)
        row["budget_percent"] = b
        all_summaries.append(row)
    all_prediction_rows.extend(fan_out_predictions(full_rows, BUDGETS))
    cleanup_model(full_model)
    save_results()

if "h2o" in POLICIES:
    for budget in BUDGETS:
        kv = budget_to_kv_settings(budget, MAX_CONTEXT_LENGTH)
        cleanup_model()
        print(
            f"\nLoading h2o model: budget={budget}% "
            f"hh_ratio={kv['hh_ratio']:.6g} h2o_full_cache_size={kv['full_cache_size']} "
            f"(~{kv['approx_effective_kv']} KV slots)"
        )
        model = load_h2o_model_ratio(MODEL_NAME_OR_PATH, kv["hh_ratio"], kv["full_cache_size"])
        summary, pred_rows = evaluate_model(
            model,
            tokenizer,
            test_data,
            policy="h2o",
            budget_percent=budget,
            hh_size=None,
            recent_size=None,
            hh_ratio=kv["hh_ratio"],
            h2o_full_cache_size=kv["full_cache_size"],
            approx_effective_kv=kv["approx_effective_kv"],
        )
        summary["total_budget_tokens"] = kv["total_budget"]
        print("summary:", summary)
        all_summaries.append(summary)
        all_prediction_rows.extend(pred_rows)
        save_results()
        cleanup_model(model)

save_results()
print("Done.")


In [ ]:
results_df = pd.DataFrame(all_summaries)
results_df = results_df.sort_values(["policy", "budget_percent"])
results_df.to_csv(OUTPUT_DIR / "results.csv", index=False)

pred_df = pd.DataFrame(all_prediction_rows)
pred_df.to_csv(OUTPUT_DIR / "predictions.csv", index=False)

with open(OUTPUT_DIR / "results.json", "w") as f:
    json.dump(all_summaries, f, indent=2)

display(results_df)

In [ ]:
plt.figure(figsize=(8, 5))

# Styles: Full = dotted horizontal baseline; Local = recent-KV (orange, circles); H2O = HH+recent (blue, stars).
POLICY_PLOT_STYLE = {
    "full": {"color": "black", "linestyle": ":", "label": "Full (baseline)", "linewidth": 2.0},
    "local": {"color": "#ff7f0e", "linestyle": "-", "marker": "o", "markersize": 5, "label": "Local (recent KV)"},
    "h2o": {"color": "#1f77b4", "linestyle": "-", "marker": "*", "markersize": 9, "label": "H2O"},
}

for policy in ["full", "local", "h2o"]:
    sub = results_df[results_df["policy"] == policy].sort_values("budget_percent")
    if len(sub) == 0:
        continue
    st = POLICY_PLOT_STYLE[policy]
    plot_kw = {
        "color": st["color"],
        "linestyle": st["linestyle"],
        "label": st["label"],
    }
    if "linewidth" in st:
        plot_kw["linewidth"] = st["linewidth"]
    if st.get("marker"):
        plot_kw["marker"] = st["marker"]
        plot_kw["markersize"] = st["markersize"]
    plt.plot(sub["budget_percent"], sub["accuracy"], **plot_kw)

# Paper-style x-axis: left = larger cache budget, right = smaller (toward 0%).
plt.gca().invert_xaxis()

plt.xlabel("KV cache budget (% of max_prompt+MAX_NEW_TOKENS on eval split; → toward 0%)")
plt.ylabel("GSM8K exact-match accuracy")
plt.title("GSM8K accuracy vs KV cache budget (cf. H2O Fig. 4: Local = most recent KV)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

plot_path = OUTPUT_DIR / "accuracy_vs_budget.png"
plt.savefig(plot_path, dpi=200)
plt.show()

print("Saved:", plot_path)


## Sanity checks

- **Budget axis**: `%` is relative to **`max_prompt_tokens + MAX_NEW_TOKENS`** on the current **eval** `test_data` (including `LIMIT`), computed after the tokenizer loads.
- **Local** is **not** comparable to vanilla full KV: it is **recent-window-only** H2O (`hh_size=0`). **Full @ 100%** (H2O ratio mode at the full reference length) is the closer check against a full-cache baseline if you add one.
- **H2O** (swept ratio) at low budgets should usually sit below Full at 100%.
- If you see `unexpected keyword argument 'padding_attention_mask'`, your `Qwen3_5DecoderLayer.forward(...)` is not forwarding custom kwargs into `self_attn`; patch the decoder layer accordingly.
- If you see KV length mismatch errors, inspect `self.kv_cache.position_ids` versus `key_states.shape[2]` in `H2OQwen3_5Attention._build_additive_attention_mask`.
